Formatar dados e preparar a base para os testes


In [ ]:
from pathlib import Path
import hashlib
import json
import numpy as np
import pandas as pd

SOURCE = Path("sabrina_original.csv")
OUT = Path("experimental_protocol")
OUT.mkdir(exist_ok=True)
assert SOURCE.exists(), "Coloque o CSV original na pasta de trabalho."
source_hash = hashlib.sha256(SOURCE.read_bytes()).hexdigest()
df = pd.read_csv(SOURCE, sep=";")
assert df.shape == (80, 25), "A base mudou: reveja o protocolo antes de continuar."
assert df["track_id"].is_unique
assert df["uri"].is_unique

for col in ["spotify_streams", "videoclip_views"]:
    df[col] = pd.to_numeric(df[col].astype(str).str.replace(".", "", regex=False), errors="raise")
df["instrumentalness"] = pd.to_numeric(
    df["instrumentalness"].astype(str).str.replace(",", ".", regex=False), errors="raise"
)
df["release_date"] = pd.to_datetime(df["release_date"], format="mixed", dayfirst=True, errors="raise")
df["duration_min"] = df["duration_ms"] / 60000
df["album_family"] = df["album"].replace({"emails i can’t send fwd": "emails i can’t send"})
FEATURES = ["duration_min", "acousticness", "danceability", "energy", "instrumentalness",
            "liveness", "loudness", "speechiness", "tempo", "valence"]
TARGET = "spotify_streams"
TEST_FAMILY = "Man’s Best Friend"
SPLIT_SEEDS = [11, 29, 47, 71, 101]
MODEL_SEEDS = [11, 29, 47]
assert all(pd.api.types.is_numeric_dtype(df[c]) for c in FEATURES)
assert not np.isinf(df[FEATURES].to_numpy(dtype=float)).any()
assert df[TARGET].notna().all() and (df[TARGET] >= 0).all()
print("Dados carregados; conversões determinísticas concluídas.")

Dados carregados; conversões determinísticas concluídas.


Separar a base entre desenvolvimento e testes


In [3]:
test_mask = df["album_family"].eq(TEST_FAMILY)
dev = df.loc[~test_mask].copy()
test_ids = df.loc[test_mask, "track_id"].tolist()
assert len(dev) == 68 and len(test_ids) == 12
assert dev["album_family"].nunique() == 6
assert set(dev["album_family"]).isdisjoint({TEST_FAMILY})

assignment = df[["track_id", "uri", "album", "album_family"]].copy()
assignment["partition"] = np.where(test_mask, "test", "development")
print("Desenvolvimento:", len(dev), "faixas;", dev["album_family"].nunique(), "famílias")
print("Teste reservado:", len(test_ids), "faixas; 1 família")
print(dev.groupby("album_family").size().rename("faixas"))

Desenvolvimento: 68 faixas; 6 famílias
Teste reservado: 12 faixas; 1 família
album_family
EVOLution              10
Eyes Wide Open         12
Short n' Sweet         12
Singular: Act I         8
Singular: Act II        9
emails i can’t send    17
Name: faixas, dtype: int64


Dividir os dados em grupos (albuns) para organizar como os modelos serao avaliados

In [4]:
groups = np.array(sorted(dev["album_family"].unique()))
fold_rows = []
for seed in SPLIT_SEEDS:
    shuffled = np.random.default_rng(seed).permutation(groups)
    for fold, val_groups in enumerate(np.array_split(shuffled, 3), start=1):
        is_val = dev["album_family"].isin(val_groups)
        train = dev.loc[~is_val]
        val = dev.loc[is_val]
        assert set(train["album_family"]).isdisjoint(set(val["album_family"]))
        assert set(train["track_id"]).isdisjoint(set(val["track_id"]))
        assert not (set(train["track_id"]) | set(val["track_id"])) & set(test_ids)
        fold_rows.append({
            "split_seed": seed, "fold": fold,
            "train_ids": train["track_id"].tolist(), "validation_ids": val["track_id"].tolist(),
            "train_groups": sorted(train["album_family"].unique().tolist()),
            "validation_groups": sorted(val["album_family"].unique().tolist()),
            "n_train": len(train), "n_validation": len(val)
        })
for seed in SPLIT_SEEDS:
    ids = [i for row in fold_rows if row["split_seed"] == seed for i in row["validation_ids"]]
    assert len(ids) == len(set(ids)) == len(dev)
fold_summary = pd.DataFrame([{k: v for k, v in row.items() if k not in ["train_ids", "validation_ids", "train_groups"]} for row in fold_rows])
print(fold_summary.to_string(index=False))
print("Partições de validação distintas:", len({tuple(row["validation_groups"]) for row in fold_rows}))

 split_seed  fold                       validation_groups  n_train  n_validation
         11     1       [Eyes Wide Open, Singular: Act I]       48            20
         11     2 [Singular: Act II, emails i can’t send]       42            26
         11     3             [EVOLution, Short n' Sweet]       46            22
         29     1            [EVOLution, Singular: Act I]       50            18
         29     2   [Short n' Sweet, emails i can’t send]       39            29
         29     3      [Eyes Wide Open, Singular: Act II]       47            21
         47     1        [EVOLution, emails i can’t send]       41            27
         47     2      [Short n' Sweet, Singular: Act II]       47            21
         47     3       [Eyes Wide Open, Singular: Act I]       48            20
         71     1 [Singular: Act II, emails i can’t send]       42            26
         71     2        [Eyes Wide Open, Short n' Sweet]       44            24
         71     3           

Definicao de modelos utilizados e estabelecer a medicao de desempenho


In [ ]:
protocol = {
    "version": "1.0", "source_sha256": source_hash,
    "n_total": len(df), "n_development": len(dev), "n_test": len(test_ids),
    "target": TARGET, "target_transform": "log1p", "features": FEATURES,
    "test_family": TEST_FAMILY, "split_seeds": SPLIT_SEEDS, "rf_seeds": MODEL_SEEDS,
    "folds_per_repeat": 3,
    "primary_metric": "MAE log1p, macro por familia; media entre repeticoes",
    "rf_params": {"n_estimators": 200, "max_depth": 3, "min_samples_leaf": 3},
    "bootstrap": {"unit": "album_family", "n_resamples": 10000, "seed": 2026,
                  "interpretation": "exploratoria; apenas seis grupos"},
    "limitations": ["EDA previa na base inteira", "origens complementares no teste",
                    "amostra nao probabilistica", "sem previsao de streams em horizonte fixo"]
}
assignment.to_csv(OUT / "partitions.csv", index=False)
(OUT / "protocol.json").write_text(json.dumps(protocol, ensure_ascii=False, indent=2), encoding="utf-8")
(OUT / "folds.json").write_text(json.dumps(fold_rows, ensure_ascii=False, indent=2), encoding="utf-8")
print("Protocolo, partições e folds registrados. Nenhum modelo foi treinado.")

Protocolo, partições e folds registrados. Nenhum modelo foi treinado.
